# Literature-derived gene descriptions

Inspect exact curated text, missing descriptions and reproducible text-model provenance before interpreting semantic similarity.

**Default sample: E7.5, replicate 1 — `GSM9046243_Embryo_E7.5_stereo_rep1.h5ad`.** Read only its ordered gene identifiers from `/var/_index`. Expression values, variable-gene flags and scaled-data flags do not contribute. Synthetic fallback is disabled; external biological priors still require reviewed resources.

Use a kernel installed from the root `requirements.txt`. This notebook uses only expression-independent sources. It does not select genes, reconstruct expression, or combine blocks. See [the implementation README](../README.md) for complete file schemas and audit requirements.


## Setup and parameters

All ten notebooks use the first E7.5 sample from the supplied `GSE278603_RAW` directory. Its full original gene index defines the shared ordered input list, including any duplicate rows. No expression-based filtering or ranking is performed. `GENE_ID_COLUMN=None` reads `var_names`; an explicit alternative must be a string identifier field.

Change `CONFIG_PATH` to a reviewed biological-resource configuration. The supplied example is an unaudited template with missing local resources, so it cannot populate real priors yet. Missing files or models are reported; synthetic data are never substituted for this sample. Inference and remote downloads require explicit opt-in.

Run `python -m preprocessing.biological_priors.prepare_all` from the repository root first. This retrieves the real source files, prepares normalized tables and caches the pretrained models in `priors.mouse.json`. Set `ENABLE_INFERENCE=True` for protein or text embeddings once preparation finishes.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "preprocessing/biological_priors/core.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from preprocessing.biological_priors.core import open_text
from preprocessing.biological_priors.notebook_support import (
    prepare_notebook, source_overview, input_preview, execute_notebook, source_report,
    block_summary, read_block, validate_exports, feature_preview, plot_block,
    covered_genes, covered_go_annotations, similar_covered_genes,
)
get_ipython().run_line_magic("matplotlib", "inline")


In [ ]:
FAMILY = 'literature'
DATASET_DIR = Path("/insomnia001/depts/morpheus/projects/hackathon/Datasets/GSE278603_RAW")
SAMPLE_PATH = DATASET_DIR / "GSM9046243_Embryo_E7.5_stereo_rep1.h5ad"
GENE_ID_COLUMN = None  # Read var_names (/var/_index); do not use variable/scaled flags.
CONFIG_PATH = REPO_ROOT / "preprocessing/results/priors.mouse.json"  # Replace with reviewed resource paths/audits.
ALLOW_SYNTHETIC = False
ENABLE_INFERENCE = False  # Local ESM/text inference; pretrained weights must already exist.
ALLOW_DOWNLOADS = False   # Explicit HTTPS/checksum descriptors only when enabled.
OUTPUT_ROOT = REPO_ROOT / "preprocessing/results/biological_priors_notebooks"
MAX_PREVIEW_GENES = 20
SHOW_MISSINGNESS_DETAILS = False  # Optional diagnostic tables.
QUERY_GENE = None  # e.g. "Sox2"; None inspects the first covered genes.
EXPORT_COVERED_GENE_LISTS = True


In [ ]:
context = prepare_notebook(
    FAMILY, config_path=CONFIG_PATH, output_root=OUTPUT_ROOT,
    sample_path=SAMPLE_PATH, gene_id_column=GENE_ID_COLUMN,
    allow_synthetic=ALLOW_SYNTHETIC, enable_inference=ENABLE_INFERENCE,
    allow_downloads=ALLOW_DOWNLOADS,
)
display(Markdown(f"**Mode: {context['mode']}**"))
print("Sample:", SAMPLE_PATH)
print("Ordered gene list:", context['genes'])
print("New run directory:", context['session'])
print("Inference sources disabled:", context['disabled_sources'])
display(pd.DataFrame([context['sample_provenance']]))
ordered_genes = context['genes'].read_text().splitlines()
print(f"{len(ordered_genes):,} input gene identifiers; first 20 shown in original order:")
display(pd.DataFrame({'input_row': range(min(20, len(ordered_genes))), 'input_id': ordered_genes[:20]}))


## Resource review and construction

Use curated mouse descriptions with their exact references. Do not generate biological filler or substitute gene-name embeddings. Real text encoding requires a pinned local model and its training audit, or reviewed precomputed vectors plus the exact description file. General text discussing biology is distinct from pretraining on expression matrices; expression-trained gene models remain prohibited.

Model pooling and truncation affect the result; record the chosen model/revision and model-default processing. Text similarity measures functional semantics, not calibrated expression predictability. Expensive/local inference is disabled by default.

**No synthetic text or vectors are substituted for this sample. Missing descriptions stay missing, and text-model inference is disabled by default.**

Resources: [MGI/Alliance reports](https://www.informatics.jax.org/downloads/reports/index.html), [Sentence Transformers models](https://sbert.net/docs/sentence_transformer/pretrained_models.html).

Keep distinct source declarations and evidence channels separate. Inspect the audit below before running construction; a database name alone is not an expression-independence review.


In [ ]:
display(source_overview(context))
for spec in context['config']['sources']:
    print(spec['name'], "AUDIT:")
    print(json.dumps(spec.get('audit', {}), indent=2))
for name, lines in input_preview(context).items():
    print("\nBounded source preview:", name)
    print("\n".join(lines))


## Build and report source availability

This calls the same strict pipeline as the command-line entry point. It exports each feature matrix and its own similarity separately. Missing files/packages/models and filtered/invalid sources are reported explicitly; no block is silently filled with invented values. If no source is configured for this family, no feature block is populated.


In [ ]:
context = execute_notebook(context)
display(source_report(context))
display(block_summary(context))
manifest = context['manifest'] or {}
blocks = [read_block(context, entry) for entry in manifest.get('blocks', [])]
if manifest and SHOW_MISSINGNESS_DETAILS:
    display(pd.read_csv(context['output'] / 'identifier_mapping.tsv', sep='	', keep_default_na=False))
    print("Family coverage:", manifest['families'][FAMILY])
if not blocks:
    print("No blocks were produced. Inspect the source report and resolve dependencies/audits.")


## Family-specific inspection

Inspect the family-specific evidence and construction choices below. Only successfully populated, reviewed sources produce blocks for the sample gene list.


In [ ]:
for block in blocks:
    p = block['provenance']; spec = p['source']
    display(pd.DataFrame([{key: spec.get(key, '') for key in ['model', 'model_revision', 'pooling', 'model_audit']}]))
    if p.get('exact_descriptions'):
        display(pd.DataFrame(p['exact_descriptions']).head(12))
    if p.get('exact_inputs_file'):
        exact = block['folder'] / p['exact_inputs_file']
        print('Exact saved description source:', exact)
        with open_text(exact) as handle:
            for _, line in zip(range(12), handle):
                print(line.rstrip()[:1500])


## Covered genes, their features and separate similarities

For every block, `K = Z @ Z.T`, where each nonzero row of `X` is L2-normalized to form `Z`. This cosine Gram kernel is PSD and ranges from -1 to 1 (0 to 1 for nonnegative features). Negative values from signed profiles are retained. Missing or unusable rows have zero features and a zero similarity row/column **including the diagonal**. Consult coverage metadata: zero similarity does not prove biological dissimilarity.

Only bounded previews are densified. Each figure represents one block; no feature concatenation or fused kernel is produced.


The default view shows genes with usable features in this specific block: names/IDs, original input positions, nonzero feature counts, and nearest covered neighbors. Feature previews select nonzero columns for the displayed genes, and heatmaps use only covered rows in their original order. The left plot shows feature counts among covered genes. Set `QUERY_GENE` to inspect a particular gene and `SHOW_MISSINGNESS_DETAILS=True` for diagnostics. Full covered-gene tables are exported separately for inspection; saved feature/similarity matrices retain the full shared gene order. Coverage can differ between blocks, and the neighbor tables are inspections, not gene-panel selection.


In [ ]:
for block in blocks:
    display(Markdown("### " + block['entry']['name']))
    covered = covered_genes(block)
    print(f"{len(covered):,} covered input genes with usable features")
    display(covered.head(MAX_PREVIEW_GENES))
    display(feature_preview(block, covered_only=True))
    print("Similar covered genes from this block's own kernel:")
    neighbors = similar_covered_genes(block, query=QUERY_GENE)
    display(neighbors)
    if QUERY_GENE is not None and neighbors.empty:
        print("No covered query/neighbor pair in this block; choose a gene from the covered table.")
    if EXPORT_COVERED_GENE_LISTS:
        inspection = context['session'] / 'covered_genes'
        inspection.mkdir(exist_ok=True)
        export = inspection / (block['entry']['name'] + '.tsv')
        covered.to_csv(export, sep='\t', index=False)
        print("Full covered-gene list:", export)
    if SHOW_MISSINGNESS_DETAILS:
        display(block['coverage'])
    plot_block(block, max_genes=MAX_PREVIEW_GENES, covered_only=True)


## Validation and exported artifacts

The pipeline checks numerical PSD, dimensions and missing-information conventions. The following cell independently checks saved row alignment, labels, finite/symmetric matrices, zero missing rows and artifact hashes, and reports the recorded numerical PSD result. The checks apply to each block separately.

Source versions/URLs, upstream audits, evidence filters, construction parameters, model revisions and input hashes are retained in the manifest and each block's provenance. These biological kernels are not calibrated expression covariance. Feature aggregation remains unimplemented and is never invoked.


In [ ]:
display(validate_exports(context))
if manifest:
    display(pd.DataFrame([{'block': entry['name'], **entry['files']} for entry in manifest['blocks']]))
    print("Manifest:", context['output'] / 'manifest.json')
print("Run/dependency report:", context['session'] / 'notebook-report.json')
print("Mode:", context['mode'])
if context['mode'] == 'SYNTHETIC DEMONSTRATION':
    print("Real biological coverage remains unassessed; no neural inference was performed.")
